![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Custom Indicator Warm-Up Research

Define a custom indicator and warm it from typed daily TradeBar history.

### Set Up QuantBook

Create a daily SPY subscription for the custom indicator updates.

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 9, 1)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY", Resolution.DAILY)

### Define Indicator

Implement annualized volatility by using [IndicatorExtensions](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/combining-indicators#08-Custom-Chains) to chain [LogReturn](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/supported-indicators/log-return) into [StandardDeviation](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/supported-indicators/standard-deviation).

In [ ]:
class CustomVolatility(PythonIndicator):

    def __init__(self, period, trading_days_per_year=252):
        super().__init__()
        self.warm_up_period = period + 1
        self._trading_days_per_year = trading_days_per_year
        self.value = 0
        self._log_return = LogReturn(1)
        self._standard_deviation = IndicatorExtensions.of(StandardDeviation(period), self._log_return)

    def update(self, input_: BaseData):
        # Annualized log-return volatility.
        price = input_.value
        if price <= 0:
            return
        self._log_return.update(input_.end_time, price)
        if self.is_ready:
            self.value = self._standard_deviation.current.value * math.sqrt(self._trading_days_per_year) * 100.0
        return self.is_ready

    @property
    def is_ready(self) -> bool:
        return self._standard_deviation.is_ready

### Build Time Series

Replay daily history and store annualized volatility values.

In [ ]:
trading_days_per_month = 21
volatility = CustomVolatility(3 * trading_days_per_month)
volatility_by_date = {
    bar.end_time: volatility.value
    for bar in qb.history[TradeBar](equity, volatility.warm_up_period + 10, Resolution.DAILY)
    if volatility.update(bar)
}

indicator_values = pd.Series(volatility_by_date, name="volatility")
indicator_values

### Signal Series

Display the 1/0 signal generated when volatility is increasing.

In [ ]:
# Check if volatility increased from the previous day.
signal = (indicator_values.diff() > 0).astype(int)
signal